In [0]:
print("hello worldcup")

hello worldcup


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

#Crée le coffre "confluent" si il n'existe pas
try:
    w.secrets.create_scope(scope="confluent")
    print("scope 'confluent' créé")
except Exception as e:
    print ("scope déjà existant: ", e)

# Range les secrets
w.secrets.put_secret(scope="confluent", key="bootstrap_servers", string_value="...")
w.secrets.put_secret(scope="confluent", key="api_key", string_value="...")
w.secrets.put_secret(scope="confluent", key="api_secret", string_value="...")

print("secrets enregistrés")
    

scope déjà existant:  Scope confluent already exists!
secrets enregistrés


In [0]:
bootstrap = dbutils.secrets.get("confluent", "bootstrap_servers")
api_key = dbutils.secrets.get("confluent", "api_key")
api_secret = dbutils.secrets.get("confluent", "api_secret")    

jaas = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="{api_key}" password="{api_secret}";'
)

kafka_options = {
    "kafka.bootstrap.servers": bootstrap,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": jaas,
    "startingOffsets": "earliest",
}

In [0]:
test_df=(
    spark.read.format("kafka")
    .options(**kafka_options)
    .option("subscribe", "match-events")
    .load()
)

(test_df
    .selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)", "topic", "partition", "offset", "timestamp")
    .show(5, truncate=False))
    

+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------+------+-----------------------+
|key               |value                                                                                                                                                                                             |topic       |partition|offset|timestamp              |
+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------+------+-----------------------+
|WC2026-FRA-BRA-001|{"match_id": "WC2026-FRA-BRA-001", "minute": 0, "event_type": "pass", "team": "Br\u00e9sil", "player_id": "BRA10", "player_name": "R. Santos", "timestamp": "2026-07-11T22

In [0]:
catalog = spark.sql("SELECT current_catalog()").collect()[0][0]

spark.sql("CREATE SCHEMA IF NOT EXISTS worldcup")
spark.sql("CREATE VOLUME IF NOT EXISTS worldcup.checkpoints")

checkpoint_base = f"/Volumes/{catalog}/worldcup/checkpoints"
print("catalog :", catalog)
print("checkpoints :", checkpoint_base)




catalog : workspace
checkpoints : /Volumes/workspace/worldcup/checkpoints


In [0]:
# Fonction d'ingestion Bronze : réutilisable pour n'importe quel topic Kafka.
# On passe en paramètres ce qui change d'un flux à l'autre (le topic et la table cible) ;
# tout le reste (connexion, mise en forme, checkpoint, trigger) est identique.
def ingest_bronze(topic: str, table: str):
    """Ingère un topic Kafka vers une table Delta Bronze (données brutes + traçabilité)."""

    # --- 1) SOURCE : lecture en continu du topic Kafka passé en paramètre ---
    stream = (
        spark.readStream.format("kafka")   # connecteur Kafka intégré au runtime
        .options(**kafka_options)          # connexion : bootstrap + SASL_SSL + secrets du coffre
        .option("subscribe", topic)        # le topic varie selon l'appel (match-events / fan-tweets)
        .load()
    )

    # --- 2) MISE EN FORME BRONZE : on garde le brut, on rend lisible + on trace ---
    bronze = (
        stream.selectExpr(
            "CAST(key AS STRING)   AS key",    # clé Kafka (binaire) -> texte
            "CAST(value AS STRING) AS value",  # JSON brut (binaire) -> texte, NON parsé (rôle du Silver)
            "topic", "partition", "offset", "timestamp",  # métadonnées Kafka conservées telles quelles
        )
        .withColumn("ingest_time", current_timestamp())  # horodatage d'ingestion (traçabilité)
    )

    # --- 3) ÉCRITURE : vers la table Delta Bronze, avec un checkpoint PROPRE à ce flux ---
    (bronze.writeStream
        .format("delta")  # format Delta = Parquet + journal transactionnel (ACID)
        # checkpoint nommé d'après la table => chaque flux a sa propre mémoire d'offsets (pas de conflit)
        .option("checkpointLocation", f"{checkpoint_base}/{table}")
        .trigger(availableNow=True)          # lit tout le dispo, écrit, puis s'arrête (mode serverless)
        .toTable(f"worldcup.{table}")        # table Delta gérée cible (schema.table)
        .awaitTermination())                 # attend la fin du micro-batch avant de rendre la main


# --- Appels : un par flux. La logique est écrite une seule fois, on l'exécute deux fois. ---
ingest_bronze("match-events", "worldcup_match_bronze")   # flux des événements de match
ingest_bronze("fan-tweets",   "worldcup_tweets_bronze")  # flux des tweets de supporters

In [0]:
# Nombre de lignes ingérées dans la table Bronze
display(spark.sql("SELECT COUNT(*) AS n FROM worldcup.worldcup_match_bronze"))

# Aperçu des derniers événements écrits (offset le plus récent en premier)
display(spark.sql("SELECT * FROM worldcup.worldcup_match_bronze ORDER BY offset DESC LIMIT 5"))
        

n
14


key,value,topic,partition,offset,timestamp,ingest_time
WC2026-FRA-BRA-001,"{""match_id"": ""WC2026-FRA-BRA-001"", ""minute"": 86, ""event_type"": ""shot"", ""team"": ""France"", ""player_id"": ""FRA05"", ""player_name"": ""O. Demb\u00e9l\u00e9"", ""timestamp"": ""2026-07-11T22:29:14.920629+00:00""}",match-events,0,13,2026-07-11T22:29:14.920Z,2026-07-12T21:48:20.168Z
WC2026-FRA-BRA-001,"{""match_id"": ""WC2026-FRA-BRA-001"", ""minute"": 78, ""event_type"": ""foul"", ""team"": ""France"", ""player_id"": ""FRA05"", ""player_name"": ""O. Demb\u00e9l\u00e9"", ""timestamp"": ""2026-07-11T22:29:09.805255+00:00""}",match-events,0,12,2026-07-11T22:29:09.805Z,2026-07-12T21:48:20.168Z
WC2026-FRA-BRA-001,"{""match_id"": ""WC2026-FRA-BRA-001"", ""minute"": 75, ""event_type"": ""pass"", ""team"": ""Br\u00e9sil"", ""player_id"": ""BRA08"", ""player_name"": ""V. Junior"", ""timestamp"": ""2026-07-11T22:29:07.358780+00:00""}",match-events,0,11,2026-07-11T22:29:07.359Z,2026-07-12T21:48:20.168Z
WC2026-FRA-BRA-001,"{""match_id"": ""WC2026-FRA-BRA-001"", ""minute"": 66, ""event_type"": ""shot"", ""team"": ""France"", ""player_id"": ""FRA02"", ""player_name"": ""S. Umtiti"", ""timestamp"": ""2026-07-11T22:29:01.607604+00:00""}",match-events,0,10,2026-07-11T22:29:01.607Z,2026-07-12T21:48:20.168Z
WC2026-FRA-BRA-001,"{""match_id"": ""WC2026-FRA-BRA-001"", ""minute"": 58, ""event_type"": ""foul"", ""team"": ""Br\u00e9sil"", ""player_id"": ""BRA08"", ""player_name"": ""V. Junior"", ""timestamp"": ""2026-07-11T22:28:56.640174+00:00""}",match-events,0,9,2026-07-11T22:28:56.640Z,2026-07-12T21:48:20.168Z


In [0]:
display(spark.sql("SELECT COUNT(*) AS n FROM worldcup.worldcup_tweets_bronze"))
display(spark.sql("SELECT * FROM worldcup.worldcup_tweets_bronze ORDER BY offset DESC LIMIT 5"))

n
43


key,value,topic,partition,offset,timestamp,ingest_time
@farii,"{""user_name"": ""@farii"", ""message"": ""Alleeeez les bleus"", ""timestamp"": ""2026-07-11T22:32:05.784882+00:00""}",fan-tweets,0,42,2026-07-11T22:32:05.785Z,2026-07-12T21:48:29.042Z
@vieillebranche,"{""user_name"": ""@vieillebranche"", ""message"": ""Alleeeez les bleus"", ""timestamp"": ""2026-07-11T22:32:03.979418+00:00""}",fan-tweets,0,41,2026-07-11T22:32:03.979Z,2026-07-12T21:48:29.042Z
@lubiel,"{""user_name"": ""@lubiel"", ""message"": ""Mais ils font quoi pur\u00e9e !"", ""timestamp"": ""2026-07-11T22:32:03.353802+00:00""}",fan-tweets,0,40,2026-07-11T22:32:03.354Z,2026-07-12T21:48:29.042Z
@luciole,"{""user_name"": ""@luciole"", ""message"": ""FAUUUTEEEE mais y'a FAUTE l\u00e0"", ""timestamp"": ""2026-07-11T22:32:02.357375+00:00""}",fan-tweets,0,39,2026-07-11T22:32:02.357Z,2026-07-12T21:48:29.042Z
@farii,"{""user_name"": ""@farii"", ""message"": ""Bravo les gars"", ""timestamp"": ""2026-07-11T22:32:01.648308+00:00""}",fan-tweets,0,38,2026-07-11T22:32:01.648Z,2026-07-12T21:48:29.042Z
